# TP3 - Exportacion para el dashboard de Power BI

Este notebook genera los archivos que despues consume Power BI. Sale referenciado desde el notebook principal (`TP3_Grupo_1.ipynb`) al cierre de la seccion 6.

Archivos que produce en `data/processed/`:

- `fact_propiedades.csv` (una fila por propiedad)
- `dim_barrios.csv` (una fila por barrio)
- `dim_clusters.csv` (una fila por cluster, con nombre y color)
- `dim_puntos_referencia.csv` (subte + tren + espacios verdes para overlay del mapa)
- `dim_coeficientes_modelo.csv` (coeficientes Ridge e importancia por permutacion)
- `barrios.geojson` (poligonos de los barrios para el Shape Map)


In [23]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

SEED = 42
np.random.seed(SEED)

CHECKPOINT_DF = '../data/processed/checkpoint_post_modelos.pkl'
CHECKPOINT_EXTRAS = '../data/processed/checkpoint_post_modelos_extras.pkl'
KPIS = '../data/raw/dataframe_kpis.tsv'
BARRIOS_GEOJSON_URL = (
    'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/'
    'ministerio-de-educacion/barrios/barrios.geojson'
)

df = pd.read_pickle(CHECKPOINT_DF)
df_kpis = pd.read_csv(KPIS, sep='\t')
with open(CHECKPOINT_EXTRAS, 'rb') as f:
    extras = pickle.load(f)

perfil_mediana       = extras.get('perfil_mediana')
perfil_cluster_geo   = extras.get('perfil_cluster_geo')
coef_ridge           = extras.get('coef_ridge')
perm_importance      = extras.get('perm_importance')
nombres_clusters     = extras.get('nombres_clusters', {})

# Capas geograficas ya procesadas por el notebook principal
verdes = extras.get('verdes')
tren   = extras.get('tren')
subte  = extras.get('subte')
gdf_barrios = extras.get('gdf_barrios')

print(f'Propiedades cargadas: {len(df):,}')
if verdes is not None: print(f'Espacios verdes:    {len(verdes)}')
if tren is not None:   print(f'Estaciones de tren: {len(tren)}')
if subte is not None:  print(f'Estaciones de subte:{len(subte)}')
if gdf_barrios is not None: print(f'Poligonos de barrios: {len(gdf_barrios)}')


Propiedades cargadas: 51,992
Espacios verdes:    2176
Estaciones de tren: 301
Estaciones de subte:90
Poligonos de barrios: 48


In [24]:
import os
from pathlib import Path

OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# fact_propiedades: tabla central, una fila por propiedad
# Calculamos algunas columnas derivadas que sirven para visuales de Power BI

# Identificador único combinando sitio y posting_id
df['propiedad_id'] = df['sitio'].astype(str) + '_' + df['posting_id'].astype(str)

# Precio por metro cuadrado relativo a la mediana del barrio (KPI 5)
mediana_m2_barrio = df.groupby('barrio_oficial')['precio_por_m2_usd'].transform('median')
df['precio_m2_relativo_barrio'] = (df['precio_por_m2_usd'] / mediana_m2_barrio).round(3)

# Banderas de cercanía para que Power BI haga comparativos rápidos
df['cerca_subte']  = (df['dist_subte_km'] < 0.5).astype('Int64')
df['cerca_verde']  = (df['dist_verde_km'] < 0.5).astype('Int64')
df['cerca_tren']   = (df['dist_tren_km'] < 1.0).astype('Int64')

# Marcamos como oportunidad las propiedades que están al menos 15 por ciento
# por debajo de la mediana de su barrio
df['es_oportunidad'] = (df['precio_m2_relativo_barrio'] < 0.85).astype('Int64')

# Columnas finales del fact
columnas_fact = [
    'propiedad_id', 'barrio_oficial', 'cluster', 'operacion', 'moneda',
    'precio', 'precio_usd', 'precio_por_m2', 'precio_por_m2_usd',
    'precio_m2_relativo_barrio',
    'm2_total', 'ambientes', 'dormitorios', 'baños', 'antiguedad_años', 'expensas',
    'lat', 'lon',
    'dist_subte_km', 'dist_verde_km', 'dist_tren_km',
    'cerca_subte', 'cerca_verde', 'cerca_tren',
    'indice_lujo', 'indice_confort', 'score_antiguedad',
    'pca_precio_sup', 'pca_antiguedad',
    'segmento_zona', 'segmento_zona_modelo',
    'es_oportunidad',
]
columnas_fact = [c for c in columnas_fact if c in df.columns]

fact_propiedades = df[columnas_fact].copy()
# Renombramos cluster a cluster_id para que el join con dim_clusters sea explícito
fact_propiedades = fact_propiedades.rename(columns={'cluster': 'cluster_id'})

ruta = OUTPUT_DIR / 'fact_propiedades.csv'
fact_propiedades.to_csv(ruta, index=False, encoding='utf-8-sig')
print(f'fact_propiedades.csv guardado: {len(fact_propiedades):,} filas, {len(fact_propiedades.columns)} columnas')

fact_propiedades.csv guardado: 51,992 filas, 32 columnas


In [25]:
# dim_barrios: una fila por barrio, con todos los KPIs y agregados

kpis_base = df_kpis.set_index('barrio_oficial') if 'barrio_oficial' in df_kpis.columns else df_kpis.copy()

# Cluster dominante por barrio (la moda)
cluster_dom = df.dropna(subset=['cluster']).groupby('barrio_oficial')['cluster'].agg(
    lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan
).rename('cluster_dominante_id')

# Agregados nuevos de espaciales e índices
agregados = df.groupby('barrio_oficial').agg(
    n_propiedades=('precio_por_m2_usd', 'size'),
    precio_m2_mediano_usd=('precio_por_m2_usd', 'median'),
    precio_venta_mediano_usd=('precio_usd', 'median'),
    dist_subte_mediana_km=('dist_subte_km', 'median'),
    dist_verde_mediana_km=('dist_verde_km', 'median'),
    dist_tren_mediana_km=('dist_tren_km', 'median'),
    nivel_socioeconomico=('nivel_socioeconomico', 'first'),
    indice_lujo_mediano=('indice_lujo', 'median'),
    indice_confort_mediano=('indice_confort', 'median'),
).round(3)

dim_barrios = kpis_base.join(agregados, how='outer').join(cluster_dom, how='left')

# Modalidad óptima a partir del índice neto (KPI 9): si > 1, conviene temporario
if 'indice_neto' in dim_barrios.columns:
    dim_barrios['modalidad_optima'] = np.where(
        dim_barrios['indice_neto'] > 1, 'temporario', 'largo_plazo'
    )
elif 'rent_neta_temp_pct' in dim_barrios.columns and 'rent_neta_lp_pct' in dim_barrios.columns:
    dim_barrios['modalidad_optima'] = np.where(
        dim_barrios['rent_neta_temp_pct'] > dim_barrios['rent_neta_lp_pct'],
        'temporario', 'largo_plazo'
    )

# Reset index para que barrio_oficial sea columna y no índice (más limpio en Power BI)
dim_barrios = dim_barrios.reset_index().rename(columns={'index': 'barrio_oficial'})

ruta = OUTPUT_DIR / 'dim_barrios.csv'
dim_barrios.to_csv(ruta, index=False, encoding='utf-8-sig')
print(f'dim_barrios.csv guardado: {len(dim_barrios):,} barrios, {len(dim_barrios.columns)} columnas')

dim_barrios.csv guardado: 49 barrios, 29 columnas


In [26]:
# dim_clusters: una fila por cluster, con su perfil, rentabilidad y color para los visuales

# Perfil promedio de cada cluster
perfil_cluster = df.dropna(subset=['cluster']).groupby('cluster').agg(
    n_propiedades=('precio_por_m2_usd', 'size'),
    precio_m2_mediano_usd=('precio_por_m2_usd', 'median'),
    precio_venta_mediano_usd=('precio_usd', 'median'),
    m2_mediano=('m2_total', 'median'),
    ambientes_mediano=('ambientes', 'median'),
    antiguedad_mediana=('antiguedad_años', 'median'),
    indice_lujo_mediano=('indice_lujo', 'median'),
    indice_confort_mediano=('indice_confort', 'median'),
    dist_subte_mediana_km=('dist_subte_km', 'median'),
    dist_verde_mediana_km=('dist_verde_km', 'median'),
    dist_tren_mediana_km=('dist_tren_km', 'median'),
).round(2)

# Rentabilidad mediana ponderada por barrio dentro de cada cluster
if 'rent_neta_lp_pct' in df_kpis.columns:
    rent_por_cluster = (
        df.dropna(subset=['cluster'])
          .merge(df_kpis.set_index('barrio_oficial')[['rent_neta_lp_pct', 'rent_neta_temp_pct']]
                  if 'barrio_oficial' in df_kpis.columns
                  else df_kpis[['rent_neta_lp_pct', 'rent_neta_temp_pct']],
                 left_on='barrio_oficial', right_index=True, how='left')
          .groupby('cluster').agg(
              rent_neta_lp_mediana_pct=('rent_neta_lp_pct', 'median'),
              rent_neta_temp_mediana_pct=('rent_neta_temp_pct', 'median'),
          ).round(2)
    )
    perfil_cluster = perfil_cluster.join(rent_por_cluster, how='left')

# Nombre comercial: la moda del cluster_nombre dentro de cada cluster
nombres = df.dropna(subset=['cluster', 'cluster_nombre']).groupby('cluster')['cluster_nombre'].agg(
    lambda s: s.mode().iloc[0] if not s.mode().empty else None
).rename('cluster_nombre')
perfil_cluster = perfil_cluster.join(nombres, how='left')

dim_clusters = perfil_cluster.reset_index().rename(columns={'cluster': 'cluster_id'})
dim_clusters['cluster_id'] = dim_clusters['cluster_id'].astype(int)

# Color en hexadecimal, alineado con la paleta que usa el notebook en los scatter y mapas.
# Esto le permite a Power BI usar los mismos colores que el notebook si configurás
# un color condicional sobre esta columna en los visuales de mapa y bar charts.
if 'PALETA_CLUSTERS' in dir():
    paleta = PALETA_CLUSTERS
else:
    paleta = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
dim_clusters['color_hex'] = [paleta[i % len(paleta)] for i in range(len(dim_clusters))]

ruta = OUTPUT_DIR / 'dim_clusters.csv'
dim_clusters.to_csv(ruta, index=False, encoding='utf-8-sig')
print(f'dim_clusters.csv guardado: {len(dim_clusters):,} clusters, {len(dim_clusters.columns)} columnas')
print(dim_clusters[['cluster_id', 'cluster_nombre', 'n_propiedades', 'precio_m2_mediano_usd', 'color_hex']].to_string(index=False))

dim_clusters.csv guardado: 4 clusters, 16 columnas
 cluster_id                                                                      cluster_nombre  n_propiedades  precio_m2_mediano_usd color_hex
          0                                                   Cluster 0: departamentos antiguos          17556                  26.12   #1f77b4
          1                                            Cluster 1: departamentos lejos del subte           4349                1885.25   #ff7f0e
          2                    Cluster 2: departamentos en barrios de alto nivel socioeconómico          18083                2131.58   #2ca02c
          3 Cluster 3: departamentos amplios, antiguos, en barrios de alto nivel socioeconómico           4150                2894.80   #d62728


In [27]:
# dim_puntos_referencia: estaciones de subte, tren y espacios verdes para overlay del mapa.
# Esta tabla no se relaciona con el fact; alimenta el visual de mapa con sus colores fijos.
# Usamos los GeoDataFrames y DataFrames que ya quedaron cargados en la sección 2
# (verdes, tren, subte), no listas hard-codeadas.

puntos = []

def _buscar_nombre(row, columnas, default):
    for c in columnas:
        if c in row.index and pd.notna(row.get(c)):
            return str(row[c])
    return default

# Espacios verdes (GeoDataFrame verdes con polígonos)
try:
    if 'verdes' in dir():
        for i, row in verdes.reset_index(drop=True).iterrows():
            centroide = row.geometry.centroid
            nombre = _buscar_nombre(
                row,
                ['nombre', 'NOMBRE', 'Nombre', 'name', 'nom_esp', 'tipo'],
                f'Espacio verde {i + 1}'
            )
            puntos.append({
                'tipo': 'espacio_verde',
                'nombre': nombre,
                'lat': centroide.y,
                'lon': centroide.x,
                'color_default': '#2ca02c'
            })
        print(f'  Espacios verdes: {sum(1 for p in puntos if p["tipo"] == "espacio_verde")}')
    else:
        print('  verdes no está en memoria, salteamos esa capa')
except Exception as e:
    print(f'  No se pudieron exportar los espacios verdes: {e}')

# Estaciones de tren (DataFrame 	ren con columnas lat_tren, lon_tren)
try:
    if 'tren' in dir():
        for i, row in tren.reset_index(drop=True).iterrows():
            nombre = _buscar_nombre(
                row,
                ['nombre', 'NOMBRE', 'Nombre', 'estacion', 'ESTACION', 'name'],
                f'Estación de tren {i + 1}'
            )
            puntos.append({
                'tipo': 'tren',
                'nombre': nombre,
                'lat': row['lat_tren'],
                'lon': row['lon_tren'],
                'color_default': '#d62728'
            })
        print(f'  Estaciones de tren: {sum(1 for p in puntos if p["tipo"] == "tren")}')
    else:
        print('  tren no está en memoria, salteamos esa capa')
except Exception as e:
    print(f'  No se pudieron exportar las estaciones de tren: {e}')

# Estaciones de subte (DataFrame subte con columnas lat_subte, lon_subte)
try:
    if 'subte' in dir():
        for i, row in subte.reset_index(drop=True).iterrows():
            nombre = _buscar_nombre(
                row,
                ['nombre', 'NOMBRE', 'Nombre', 'estacion', 'ESTACION', 'name'],
                f'Estación de subte {i + 1}'
            )
            puntos.append({
                'tipo': 'subte',
                'nombre': nombre,
                'lat': row['lat_subte'],
                'lon': row['lon_subte'],
                'color_default': '#1f77b4'
            })
        print(f'  Estaciones de subte: {sum(1 for p in puntos if p["tipo"] == "subte")}')
    else:
        print('  subte no está en memoria, salteamos esa capa')
except Exception as e:
    print(f'  No se pudieron exportar las estaciones de subte: {e}')

dim_puntos = pd.DataFrame(puntos)
dim_puntos.insert(0, 'punto_id', range(1, len(dim_puntos) + 1))

ruta = OUTPUT_DIR / 'dim_puntos_referencia.csv'
dim_puntos.to_csv(ruta, index=False, encoding='utf-8-sig')
print(f'\ndim_puntos_referencia.csv guardado: {len(dim_puntos):,} puntos')
print(dim_puntos['tipo'].value_counts().to_string())

  Espacios verdes: 2176
  Estaciones de tren: 301
  Estaciones de subte: 90

dim_puntos_referencia.csv guardado: 2,567 puntos
tipo
espacio_verde    2176
tren              301
subte              90


In [28]:
# dim_coeficientes_modelo: peso de cada variable sobre el precio por metro cuadrado.
# Entrenamos un Ridge sobre el log del precio por metro cuadrado para tener coeficientes
# interpretables y exportables, sin depender del estado de la sección 6.

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

features = [
    'm2_total', 'antiguedad_años',
    'indice_lujo', 'indice_confort', 'score_antiguedad',
    'dist_subte_km', 'dist_verde_km', 'dist_tren_km',
    'nivel_socioeconomico'
]
features = [f for f in features if f in df.columns and df[f].notna().sum() >= 100]

df_mod = df[
    (df['operacion'] == 'venta') &
    (df['moneda'] == 'usd') &
    df['precio_por_m2_usd'].notna() &
    df['cluster'].notna()
][features + ['cluster', 'precio_por_m2_usd']].dropna().copy()

X = pd.get_dummies(df_mod[features + ['cluster']], columns=['cluster'], drop_first=True)
y = np.log1p(df_mod['precio_por_m2_usd'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

ridge = Ridge(alpha=1.0, random_state=SEED).fit(X_train_sc, y_train)
perm = permutation_importance(ridge, X_test_sc, y_test, n_repeats=5, random_state=SEED, n_jobs=-1)

dim_coef = pd.DataFrame({
    'variable': X.columns,
    'tipo_modelo': 'ridge_precio_m2',
    'coeficiente': ridge.coef_.round(4),
    'importancia_permutacion': perm.importances_mean.round(4),
})
dim_coef['signo'] = np.where(dim_coef['coeficiente'] > 0, 'positivo', 'negativo')
dim_coef['coeficiente_abs'] = dim_coef['coeficiente'].abs()
dim_coef = dim_coef.sort_values('coeficiente_abs', ascending=False).drop(columns='coeficiente_abs')

ruta = OUTPUT_DIR / 'dim_coeficientes_modelo.csv'
dim_coef.to_csv(ruta, index=False, encoding='utf-8-sig')
print(f'dim_coeficientes_modelo.csv guardado: {len(dim_coef):,} variables')
print(dim_coef.to_string(index=False))

dim_coeficientes_modelo.csv guardado: 12 variables
            variable     tipo_modelo  coeficiente  importancia_permutacion    signo
    score_antiguedad ridge_precio_m2      -0.7571                   7.8624 negativo
     antiguedad_años ridge_precio_m2       0.5494                   4.1375 positivo
nivel_socioeconomico ridge_precio_m2       0.0499                   0.0364 positivo
         indice_lujo ridge_precio_m2       0.0203                   0.0058 positivo
         cluster_1.0 ridge_precio_m2       0.0158                   0.0039 positivo
         cluster_3.0 ridge_precio_m2       0.0113                   0.0016 positivo
        dist_tren_km ridge_precio_m2      -0.0070                   0.0009 negativo
       dist_subte_km ridge_precio_m2      -0.0059                   0.0004 negativo
         cluster_2.0 ridge_precio_m2       0.0054                   0.0006 positivo
       dist_verde_km ridge_precio_m2       0.0050                   0.0003 positivo
            m2_total ridg

In [29]:
# barrios.geojson: la geometría de los barrios para los Shape Maps de Power BI
gdf_export = gdf_barrios[['barrio_oficial', 'geometry']].copy()
ruta_geo = OUTPUT_DIR / 'barrios.geojson'
gdf_export.to_file(ruta_geo, driver='GeoJSON')
print(f'barrios.geojson guardado: {len(gdf_export):,} polígonos')

print('\nArchivos listos para Power BI:')
for nombre in ['fact_propiedades.csv', 'dim_barrios.csv', 'dim_clusters.csv',
                'dim_puntos_referencia.csv', 'dim_coeficientes_modelo.csv', 'barrios.geojson']:
    archivo = OUTPUT_DIR / nombre
    if archivo.exists():
        kb = archivo.stat().st_size / 1024
        print(f'  {nombre:<35}  {kb:>8,.1f} KB')

barrios.geojson guardado: 48 polígonos

Archivos listos para Power BI:
  fact_propiedades.csv                 16,345.7 KB
  dim_barrios.csv                          10.6 KB
  dim_clusters.csv                          0.8 KB
  dim_puntos_referencia.csv               213.7 KB
  dim_coeficientes_modelo.csv               0.7 KB
  barrios.geojson                         719.3 KB
